In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import joblib
import warnings


def main():
    warnings.filterwarnings('ignore')

    #Data Read
    file_path = 'heart_disease_uci.csv'
    df = pd.read_csv(file_path, na_values='?')

    # Handle Missing Values
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                df[col].fillna(df[col].mode()[0], inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)

    # Data Encoding
    df_processed = df.drop(['id', 'dataset'], axis=1)
    df_processed.rename(columns={'num': 'target'}, inplace=True)
    df_processed['target'] = (df_processed['target'] > 0).astype(int)
    categorical_cols = df_processed.select_dtypes(include=['object']).columns
    df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

    # Separate features (X) and target (y)
    X = df_processed.drop('target', axis=1)
    y = df_processed['target']

    # split Data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Train and Evaluate
    baseline_model = SVC(probability=True, random_state=42)
    baseline_model.fit(X_train_scaled, y_train)
    y_pred_baseline = baseline_model.predict(X_test_scaled)
    baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
    baseline_auc = roc_auc_score(y_test, baseline_model.predict_proba(X_test_scaled)[:, 1])

    #Train and Evaluate Optimized SVM Model
    best_params = {'C': 15, 'gamma': 0.05, 'kernel': 'rbf'}
    optimized_model = SVC(probability=True, random_state=42, **best_params)
    optimized_model.fit(X_train_scaled, y_train)
    y_pred_optimized = optimized_model.predict(X_test_scaled)
    optimized_accuracy = accuracy_score(y_test, y_pred_optimized)
    improvement = optimized_accuracy - baseline_accuracy

    #Final Evaluation Metrics
    print("\n-----------------------------------------")
    print("   Final Model Evaluation Metrics")
    print("-----------------------------------------")
    print(f"Chosen Model: Support Vector Machine (SVM)")
    print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
    print(f"Optimized Accuracy: {optimized_accuracy:.4f}")
    print(f"Improvement: {improvement:.4f}")
    print(f"AUC Score (Baseline): {baseline_auc:.4f}")
    print("-----------------------------------------\n")

    model_filename = 'final_model.pkl'
    joblib.dump(optimized_model, model_filename)
    scaler_filename = 'scaler.pkl'
    joblib.dump(scaler, scaler_filename)


if __name__ == "__main__":
    main()

